In [1]:
import dspy

# Konfiguration des lokalen Sprachmodells
local_llm = dspy.LM(
    "openai/Qwen3-VL-8B-Instruct-Q4_K_M.gguf", 
    api_base="http://localhost:8080/v1", 
    api_key="no_key_needed",
    temperature=2,
    cache=False
)

dspy.configure(lm=local_llm)

In [2]:
# Signatur und Modul (aus Tag 8/9)
class SentimentSignature(dspy.Signature):
    """Klassifiziert den Sentiment eines gegebenen Textes als positiv, negativ oder neutral."""
    text = dspy.InputField(desc="Der zu klassifizierende Text.")
    sentiment = dspy.OutputField(desc="Das Ergebnis der Klassifizierung: positiv, negativ oder neutral.")

class SimpleClassifier(dspy.Module):
    def __init__(self):
        super().__init__()
        self.predictor = dspy.Predict(SentimentSignature)

    def forward(self, text):
        return self.predictor(text=text)

# Definition der Beispieldaten
raw_data = [
    ("Die Bildqualität ist hervorragend und die Bedienung intuitiv.", "positiv"),
    ("Ich bin sehr zufrieden mit dem Produkt, es übertrifft meine Erwartungen.", "positiv"),
    ("Das Preis-Leistungs-Verhältnis ist unschlagbar.", "positiv"),
    ("Leider hat das Gerät nach kurzer Zeit den Geist aufgegeben.", "negativ"),
    ("Der Kundenservice war überhaupt nicht hilfreich und unfreundlich.", "negativ"),
    ("Die Akkulaufzeit ist enttäuschend kurz.", "negativ"),
    ("Das Produkt wurde pünktlich geliefert.", "neutral"),
    ("Die Verpackung war angemessen.", "neutral"),
]
trainset = [dspy.Example(text=t, sentiment=s).with_inputs("text") for t, s in raw_data]

In [3]:
# Instanziierung des unoptimierten Modells
unoptimized_classifier = SimpleClassifier()

# Ausführung einer Vorhersage, um den Prompt zu inspizieren
test_text = "Der Stoff fühlt sich angenehm an, aber der Schnitt ist seltsam."
unoptimized_classifier(text=test_text)

Prediction(
    sentiment='neutral'
)

In [4]:
# Anzeige des letzten Prompts an das LLM
local_llm.inspect_history(n=1) # Führen Sie dies in einer interaktiven Umgebung aus





[2025-11-13T12:03:36.663404]

System message:

Your input fields are:
1. `text` (str): Der zu klassifizierende Text.
Your output fields are:
1. `sentiment` (str): Das Ergebnis der Klassifizierung: positiv, negativ oder neutral.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## text ## ]]
{text}

[[ ## sentiment ## ]]
{sentiment}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Klassifiziert den Sentiment eines gegebenen Textes als positiv, negativ oder neutral.


User message:

[[ ## text ## ]]
Der Stoff fühlt sich angenehm an, aber der Schnitt ist seltsam.

Respond with the corresponding output fields, starting with the field `[[ ## sentiment ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## sentiment ## ]]
neutral

[[ ## completed ## ]]







In [5]:
from dspy.teleprompt import BootstrapFewShot

# Metrik für die Optimierung
def metric(gold, pred, trace=None):
    return gold.sentiment.lower() == pred.sentiment.lower()

# Konfiguration und Ausführung des Optimizers
optimizer = BootstrapFewShot(metric=metric, max_bootstrapped_demos=2)
optimized_classifier = optimizer.compile(SimpleClassifier(), trainset=trainset)

# Ausführung mit demselben Testtext
optimized_classifier(text=test_text)


 25%|█████████████████████                                                               | 2/8 [00:01<00:03,  1.95it/s]


Bootstrapped 2 full traces after 2 examples for up to 1 rounds, amounting to 2 attempts.


Prediction(
    sentiment='neutral'
)

In [6]:

# Anzeige des optimierten Prompts
local_llm.inspect_history(n=1) # Führen Sie dies in einer interaktiven Umgebung aus





[2025-11-13T12:03:40.094281]

System message:

Your input fields are:
1. `text` (str): Der zu klassifizierende Text.
Your output fields are:
1. `sentiment` (str): Das Ergebnis der Klassifizierung: positiv, negativ oder neutral.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## text ## ]]
{text}

[[ ## sentiment ## ]]
{sentiment}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Klassifiziert den Sentiment eines gegebenen Textes als positiv, negativ oder neutral.


User message:

[[ ## text ## ]]
Die Bildqualität ist hervorragend und die Bedienung intuitiv.


Assistant message:

[[ ## sentiment ## ]]
positiv

[[ ## completed ## ]]


User message:

[[ ## text ## ]]
Ich bin sehr zufrieden mit dem Produkt, es übertrifft meine Erwartungen.


Assistant message:

[[ ## sentiment ## ]]
positiv

[[ ## completed ## ]]


User message:

[[ ## text ## ]]
Das Preis-Leistungs-Verhältnis ist unschlagbar.


Assist